# Brain-1: Kaggle CPU / NVIDIA training

Run cells in order. The default `1 × 32 × 1024 = 32,768` tokens per optimizer step is an **unmeasured low-memory starting point**, not a promise that a roughly 15 GB GPU fits. CPU uses the same batch contract. For a separate CPU smoke run, explicitly choose a new run name, shorter sequence and fewer steps before preparing data. Nothing rescales on resume.

Synthetic data is smoke-only. Charlevel tokenization is not meaningful language-model evaluation or comparable to trained BPE. No GPU performance or hosted persistence has been validated by editing this notebook.

Kaggle working files are not automatically restored in a new session. Save outputs yourself, explicitly attach them next time, and set `RESUME_FROM` plus `PACKED_DIR` (and the same tokenizer). Only load checkpoints you trust. Preserve the run's `notebook_run.json` and logs alongside checkpoints; never put artifacts in git.

In [ ]:
from pathlib import Path

WORK = Path("/kaggle/working")
REPO_URL = "https://github.com/AHK811/Brain-1.git"
RUN_NAME = "brainlab-run-001"
DEVICE_REQUEST = "auto"
DTYPE = "auto"
MICRO_BATCH = 1
ACCUM = 32
SEQ_LEN = 1024
GRAD_CKPT = True
SEED = 0
TOTAL_STEPS = 8000
SESSION_MINUTES = 240
EVAL_EVERY = 250
EVAL_BATCHES = 64
SAVE_EVERY = 250
LOG_EVERY = 10
KEEP = 3
DATA_MODE = "synthetic"
CORPUS_DIR = None
PACKED_DIR = None
TOKENIZER_DIR = None
TOKENIZER_CORPUS = None
SYNTHETIC_WORDS = 400000
ALLOW_SYNTHETIC_MATCHES = True
VAL_RATIO = 0.05
DOC_SEP = "blank"
RESUME_FROM = None
RUN_OVERFIT = True
OVERFIT_STEPS = 120
OVERFIT_SEQ_LEN = 128
RUN_PROBE = True
PROBE_STEPS = 3
PPL_BATCHES = None
COMPARE_FP16 = False
GENERATE_TOKENS = 96
PROMPTS = ("Once upon a time", "The little girl was")
USE_DRIVE = False
DRIVE_RUN_NAME = "brainlab-run-001"
DRIVE_BASE = Path("/content/drive/MyDrive/brainlab")
RESTORE_FROM_DRIVE = None
BACKUP_SECONDS = 300

In [ ]:
import hashlib
import json
import math
import os
import re
import statistics
import subprocess
import sys
import tempfile
import time


def run(args, cwd=None, check=True, env=None):
    proc = subprocess.run([str(arg) for arg in args], cwd=None if cwd is None else str(cwd),
                          env=env, capture_output=True, text=True)
    if check and proc.returncode:
        print(proc.stdout)
        print(proc.stderr, file=sys.stderr)
        proc.check_returncode()
    return proc


def ensure_checkout(repo, url):
    repo = Path(repo).resolve()
    required = ("python/brainlab/cli.py", "scripts/setup_colab.sh", "spec/model_config.toml")
    if not repo.exists() or (repo.is_dir() and not any(repo.iterdir())):
        repo.parent.mkdir(parents=True, exist_ok=True)
        run(["git", "clone", "--depth", "1", url, repo], cwd=None)
    if not repo.is_dir() or not all((repo / name).is_file() for name in required):
        raise RuntimeError(f"Not a Brain-1 checkout: {repo}; move it aside explicitly")
    top = run(["git", "-C", repo, "rev-parse", "--show-toplevel"]).stdout.strip()
    if Path(top).resolve() != repo:
        raise RuntimeError(f"Checkout root mismatch: {top}")
    return run(["git", "-C", repo, "rev-parse", "HEAD"]).stdout.strip()


def py(code, *args):
    return run([PY, "-c", code, *args], cwd=REPO)


def read_rows(directory):
    path = Path(directory) / "train_log.jsonl"
    rows = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    if not rows or not all(isinstance(row, dict) for row in rows):
        raise RuntimeError(f"Invalid or empty authoritative log: {path}")
    return rows


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def checkpoint_name(name):
    return name in {"last.pt", "best.pt"} or re.fullmatch(r"step-[0-9]+\.pt", name) is not None


def latest_cursor(directory):
    proc = py("from pathlib import Path; import json, sys; "
              "from brainlab.train.checkpoint import latest_checkpoint; "
              "p = latest_checkpoint(Path(sys.argv[1])); "
              "print(json.dumps(str(p.resolve()) if p is not None else None))", directory)
    value = json.loads(proc.stdout)
    if value is None:
        return None
    path = Path(value)
    if not path.is_file() or path.parent.resolve() != Path(directory).resolve() or not checkpoint_name(path.name) or path.name == "best.pt":
        raise RuntimeError(f"Invalid shared checkpoint cursor: {path}")
    return path

In [ ]:
if DEVICE_REQUEST not in {"auto", "cpu", "cuda"} or DTYPE not in {"auto", "fp16", "bf16", "fp32"}:
    raise ValueError("Choose a supported device and dtype in the config cell")
if not RUN_NAME or Path(RUN_NAME).name != RUN_NAME or RUN_NAME in {".", ".."}:
    raise ValueError("RUN_NAME must be one directory name")
for value in (MICRO_BATCH, ACCUM, SEQ_LEN, TOTAL_STEPS, EVAL_EVERY, EVAL_BATCHES, SAVE_EVERY,
              LOG_EVERY, KEEP, OVERFIT_STEPS, OVERFIT_SEQ_LEN, PROBE_STEPS, GENERATE_TOKENS):
    if type(value) is not int or value < 1:
        raise ValueError("Batch, length, step and interval settings must be positive integers")
if SEQ_LEN < 2 or SEED < 0 or not math.isfinite(SESSION_MINUTES) or SESSION_MINUTES <= 0:
    raise ValueError("Invalid sequence length, seed or session budget")
if not math.isfinite(BACKUP_SECONDS) or BACKUP_SECONDS <= 0:
    raise ValueError("BACKUP_SECONDS must be positive")
if DATA_MODE not in {"synthetic", "corpus"} or not 0 < VAL_RATIO < 1:
    raise ValueError("Choose DATA_MODE synthetic/corpus and a validation ratio between zero and one")
if DATA_MODE == "corpus" and CORPUS_DIR is None and PACKED_DIR is None:
    raise ValueError("Real data requires an explicit CORPUS_DIR or PACKED_DIR")
if DATA_MODE == "synthetic" and CORPUS_DIR is not None:
    raise ValueError("Remove CORPUS_DIR or explicitly choose DATA_MODE='corpus'")
WORK = WORK.expanduser().resolve()
REPO = WORK / "Brain-1"
CKPT = WORK / "runs" / RUN_NAME
PY = Path(sys.executable)
print("checkout:", ensure_checkout(REPO, REPO_URL))
print("selected interpreter:", PY)
print("tokens per optimizer step:", MICRO_BATCH * ACCUM * SEQ_LEN)

## Environment

The checked setup uses the notebook interpreter explicitly, skips Rust/Mojo installation, and requires no separate Triton installation. Reusing a validated checkout never pulls silently. Package imports, packing, training and evaluation run through the same selected Python, not imports in the notebook kernel.

In [ ]:
setup_env = dict(os.environ, BRAINLAB_ROOT=str(REPO), BRAINLAB_PYTHON=str(PY),
                 BRAINLAB_DEVICE=DEVICE_REQUEST, BRAINLAB_SKIP_RUST="1", BRAINLAB_MOJO_INSTALL="0")
print(run(["bash", REPO / "scripts/setup_colab.sh"], cwd=REPO, env=setup_env).stdout)
print(py("import torch, numpy; print(torch.__version__, torch.version.cuda, numpy.__version__)").stdout)

## Device and parity checks

`auto` means CPU fp32, or native CUDA bf16 where supported, otherwise CUDA fp16. An explicit unavailable CUDA request fails. Gradient checkpointing is enabled by default and also usable on CPU. Trainer log headers remain authoritative for the resolved dtype. CPU checks say nothing about GPU throughput, memory fit or mixed precision.

In [ ]:
hardware = json.loads(py("""
import json, sys, torch
from brainlab.train.train import parse_args, resolve_device, resolve_dtype
request, dtype = sys.argv[1:]
parse_args(['--device', request, '--dtype', dtype])
device = resolve_device(request)
resolved = resolve_dtype(dtype, device)
print(json.dumps({'device': str(device), 'dtype': resolved, 'torch': torch.__version__,
                  'name': torch.cuda.get_device_name(device) if device.type == 'cuda' else 'CPU',
                  'memory_gib': torch.cuda.get_device_properties(device).total_memory / 2**30 if device.type == 'cuda' else None}))
""", DEVICE_REQUEST, DTYPE).stdout)
DEVICE = hardware["device"]
print(json.dumps(hardware, indent=2))
print(run([PY, "-m", "pytest", "-q"], cwd=REPO).stdout)
print("Required tests exited zero; inspect skips rather than treating them as backend validation.")

## Persistence helpers

Only completed trainer checkpoints, logs, run provenance and scored exports are eligible. SHA256 detects same-size changes; verified copies replace staged `.partial` files. Logs use fixed-length snapshots. After a successful backup, only the newest `KEEP` numbered checkpoints are retained in that verified run folder; `best.pt`, `last.pt` and unrelated files are preserved.

Explicit restores preserve logs and reject conflicting local runs. The shared trainer selector uses checkpoint steps, not special priority for a stale `last.pt`. Copies remain best-effort, not a guarantee against preemption or a storage outage.

In [ ]:
def atomic_copy(source, target):
    source, target = Path(source), Path(target)
    if source.resolve() == target.resolve():
        return False
    target.parent.mkdir(parents=True, exist_ok=True)
    staging = target.with_name(target.name + ".partial")
    try:
        with source.open("rb") as handle:
            length = os.fstat(handle.fileno()).st_size
            remaining = length
            digest = hashlib.sha256()
            while remaining:
                block = handle.read(min(1 << 20, remaining))
                if not block:
                    raise RuntimeError(f"Source shortened during snapshot: {source}")
                digest.update(block)
                remaining -= len(block)
            expected = digest.hexdigest()
            if target.is_file() and sha256(target) == expected:
                return False
            handle.seek(0)
            remaining = length
            with staging.open("wb") as output:
                while remaining:
                    block = handle.read(min(1 << 20, remaining))
                    if not block:
                        raise RuntimeError(f"Source shortened during copy: {source}")
                    output.write(block)
                    remaining -= len(block)
                output.flush()
                os.fsync(output.fileno())
        if sha256(staging) != expected:
            raise RuntimeError(f"Source changed during copy: {source}")
        staging.replace(target)
        return True
    finally:
        staging.unlink(missing_ok=True)


def artifact_paths(directory, include_exports=False):
    return sorted(path for path in Path(directory).iterdir() if path.is_file() and (
        checkpoint_name(path.name) or path.name in {"train_log.jsonl", "train_stdout.log", "notebook_run.json"}
        or (include_exports and path.name == "scored.safetensors")))


def check_backup_destination(source, destination):
    source, destination = Path(source).resolve(), Path(destination).resolve()
    if source == destination or source in destination.parents or destination in source.parents:
        raise RuntimeError("Backup and local run directories must be separate")
    contract = destination / "notebook_run.json"
    if destination.exists() and any(destination.iterdir()):
        if not contract.is_file() or sha256(contract) != sha256(source / contract.name):
            raise RuntimeError(f"Backup folder belongs to another or unverified run: {destination}")


def backup_run(source, destination, include_exports=False):
    if destination is None:
        return []
    source, destination = Path(source), Path(destination)
    check_backup_destination(source, destination)
    destination.mkdir(parents=True, exist_ok=True)
    files = artifact_paths(source, include_exports)
    files.sort(key=lambda path: (path.name != "notebook_run.json", path.name))
    copied = []
    for item in files:
        try:
            if atomic_copy(item, destination / item.name):
                copied.append(item.name)
        except FileNotFoundError:
            if item.exists():
                raise
    numbered = sorted((path for path in destination.glob("step-*.pt")
                       if path.is_file() and re.fullmatch(r"step-[0-9]+\.pt", path.name)),
                      key=lambda path: int(path.stem.split("-")[1]))
    for old in numbered[:-KEEP]:
        old.unlink()
    print("backup:", destination, copied)
    return copied


def restore_run(source, destination):
    source, destination = Path(source).expanduser().resolve(), Path(destination).resolve()
    if source.is_file():
        if not checkpoint_name(source.name):
            raise RuntimeError(f"Not a completed checkpoint: {source}")
        chosen, parent = source, source.parent
        files = [path for path in artifact_paths(parent) if not checkpoint_name(path.name) or path == source]
    elif source.is_dir():
        parent = source
        chosen = latest_cursor(parent)
        files = artifact_paths(parent, include_exports=True)
    else:
        raise FileNotFoundError(source)
    if chosen is None:
        raise RuntimeError(f"No completed resume cursor in {source}")
    if parent == destination:
        return chosen
    if parent in destination.parents or destination in parent.parents:
        raise RuntimeError("Restore source and destination must be separate directories")
    expected = {path.name: path for path in files}
    if destination.exists():
        for target in destination.iterdir():
            if target.name not in expected or not target.is_file() or sha256(target) != sha256(expected[target.name]):
                raise RuntimeError(f"Conflicting existing local run: {target}")
    destination.mkdir(parents=True, exist_ok=True)
    for item in files:
        atomic_copy(item, destination / item.name)
    return destination / chosen.name


def verify_cli_resume(checkpoint, contract):
    saved = json.loads(py("""
import json, sys
from pathlib import Path
from brainlab.train.checkpoint import load_payload
from brainlab.spec.config import load_model_spec
payload = load_payload(Path(sys.argv[1]))
if payload['spec'] != load_model_spec().to_dict():
    raise ValueError('Checkpoint model spec differs from this checkout')
print(json.dumps(payload.get('cfg', {})))
""", checkpoint).stdout)
    expected = {key: contract[key] for key in ("micro_batch", "seq_len", "seed", "total_steps", "eval_batches",
                                              "train_sequences", "val_sequences")}
    expected.update(accum_steps=contract["accum"], eval_batch_size=contract["micro_batch"], optimizer="muon")
    for key, value in expected.items():
        if key not in saved or saved[key] != value:
            raise RuntimeError(f"Cannot verify CLI checkpoint {key}: saved {saved.get(key)!r}, requested {value!r}")
    print("CLI settings verified; historical data/tokenizer hashes are unavailable. Supply the original shards and tokenizer.")


def verify_contract(directory, contract, resume=None):
    directory = Path(directory)
    path = directory / "notebook_run.json"
    if path.exists():
        if json.loads(path.read_text(encoding="utf-8")) != contract:
            raise RuntimeError("Resume data/tokenizer/batching/seed/schedule/evaluation contract changed; use a new run")
        return
    if directory.exists() and any(directory.iterdir()):
        if resume is None:
            raise RuntimeError("Existing run has no notebook provenance; use an explicit RESUME_FROM to import it")
        verify_cli_resume(resume, contract)
    directory.mkdir(parents=True, exist_ok=True)
    staging = path.with_name(path.name + ".partial")
    staging.write_text(json.dumps(contract, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    staging.replace(path)

## Explicit data and restore inputs

Select `DATA_MODE='corpus'` and a dedicated `CORPUS_DIR` for real text; no personal or attached folders are searched automatically. `TOKENIZER_DIR` selects trained BPE, with optional `TOKENIZER_CORPUS` for split-overlap checks. The default is explicitly synthetic smoke data.

`PACKED_DIR` reuses existing shards without writing to them; otherwise preparation needs an empty destination. Preserve shards and tokenizer when moving hosts. Notebook runs carry a content-hashed contract that checks future resumes even when paths change.

To import a trusted command-line run, set `RESUME_FROM` to its completed checkpoint and use the original packed data, tokenizer and run settings. New CLI checkpoints carry the settings needed for validation; older checkpoints missing those fields are rejected. The first import cannot retrospectively verify data/tokenizer hashes, so it prints that limitation and records their current hashes for later resumes.

In [ ]:
BACKUP_DIR = None
if USE_DRIVE or RESTORE_FROM_DRIVE is not None:
    raise ValueError("This Kaggle lane uses explicit attached RESUME_FROM paths, not Drive mounting")
RESTORED_CURSOR = restore_run(RESUME_FROM, CKPT) if RESUME_FROM is not None else None
print("explicit restore:", RESTORED_CURSOR or "none; existing local run will use the shared cursor")

### Preparation and validation capacity

Actual `TokenShards` lengths, not nominal report counts, determine the validation capacity. Two nonempty, disjoint evaluation windows must fit. The requested batch count is clamped to `n_val // (2 * MICRO_BATCH)`; a capacity below one fails.

In [ ]:
DATA = Path(PACKED_DIR).expanduser().resolve() if PACKED_DIR is not None else WORK / "data" / RUN_NAME
TOKENIZER_ARGS = ["--tokenizer-dir", Path(TOKENIZER_DIR).expanduser().resolve()] if TOKENIZER_DIR is not None else ["--charlevel"]
if PACKED_DIR is None:
    if DATA.exists() and any(DATA.iterdir()):
        raise RuntimeError(f"Refusing to overwrite {DATA}; explicitly set PACKED_DIR to reuse it")
    corpus_args = ["--synthetic", SYNTHETIC_WORDS] if DATA_MODE == "synthetic" else ["--corpus", Path(CORPUS_DIR).expanduser().resolve()]
    tokenizer_corpus_args = ["--tokenizer-corpus", Path(TOKENIZER_CORPUS).expanduser().resolve()] if TOKENIZER_CORPUS is not None else []
    print(run([PY, "-m", "brainlab.data.prepare", *corpus_args, *TOKENIZER_ARGS, *tokenizer_corpus_args,
               "--out", DATA, "--seq-len", SEQ_LEN, "--seed", SEED, "--val-ratio", VAL_RATIO,
               "--doc-sep", DOC_SEP, "--engine", "python"], cwd=REPO).stdout)
else:
    print("Reusing prepared data without modification:", DATA)
print("SMOKE ONLY: synthetic data" if DATA_MODE == "synthetic" else "Explicit real corpus mode")
if TOKENIZER_DIR is None:
    print("CHARLEVEL: metrics are not meaningful language-model evaluation")

In [ ]:
def inspect_data(directory, seq_len, tokenizer_dir):
    proc = py("""
import hashlib, json, sys
from pathlib import Path
from brainlab.data.shards import TokenShards
from brainlab.data.prepare import byte_tokenizer
from brainlab.tok.reference import Tokenizer
root, seq_len, tok_dir = Path(sys.argv[1]), int(sys.argv[2]), sys.argv[3]
train, val = TokenShards(root / 'train', seq_len), TokenShards(root / 'val', seq_len)
digest = hashlib.sha256()
for split in ('train', 'val'):
    for path in sorted((root / split).iterdir()):
        if path.suffix not in {'.bin', '.json'}:
            continue
        digest.update(path.relative_to(root).as_posix().encode())
        file_digest = hashlib.sha256()
        with path.open('rb') as handle:
            for block in iter(lambda: handle.read(1 << 20), b''):
                file_digest.update(block)
        digest.update(file_digest.digest())
tok = Tokenizer.load(Path(tok_dir)) if tok_dir else byte_tokenizer()
tokenizer = {'tokens': [token.hex() for token in tok.token_bytes], 'merges': tok.merges,
             'eos': tok.eos_id, 'pad': tok.pad_id}
print(json.dumps({'train': len(train), 'val': len(val), 'data_sha256': digest.hexdigest(),
                  'tokenizer_sha256': hashlib.sha256(json.dumps(tokenizer, sort_keys=True).encode()).hexdigest()}))
""", directory, seq_len, str(Path(tokenizer_dir).expanduser().resolve()) if tokenizer_dir is not None else "")
    return json.loads(proc.stdout)


def eval_capacity(n_train, n_val, micro_batch, requested):
    if min(micro_batch, requested) < 1 or n_train < micro_batch:
        raise RuntimeError("Insufficient training sequences or invalid batching")
    capacity = n_val // (2 * micro_batch)
    if capacity < 1:
        raise RuntimeError("Validation cannot hold two disjoint nonempty evaluation windows")
    return min(requested, capacity)


prepared = inspect_data(DATA, SEQ_LEN, TOKENIZER_DIR)
VAL_BATCHES = eval_capacity(prepared["train"], prepared["val"], MICRO_BATCH, EVAL_BATCHES)
print("Actual sequences:", prepared["train"], prepared["val"], "eval batches:", VAL_BATCHES)

In [ ]:
RUN_CONTRACT = {
    "version": 1, "run_name": RUN_NAME, "data_mode": DATA_MODE,
    "data_sha256": prepared["data_sha256"], "tokenizer_sha256": prepared["tokenizer_sha256"],
    "spec_sha256": sha256(REPO / "spec/model_config.toml"),
    "train_sequences": prepared["train"], "val_sequences": prepared["val"],
    "micro_batch": MICRO_BATCH, "accum": ACCUM, "seq_len": SEQ_LEN, "seed": SEED,
    "total_steps": TOTAL_STEPS, "eval_every": EVAL_EVERY, "eval_batches": VAL_BATCHES,
    "requested_eval_batches": EVAL_BATCHES,
}
verify_contract(CKPT, RUN_CONTRACT, RESTORED_CURSOR)
if BACKUP_DIR is not None:
    check_backup_destination(CKPT, BACKUP_DIR)
print(json.dumps(RUN_CONTRACT, indent=2))

### Leakage gate

Real-data matches fail. Template matches are allowed only when both synthetic mode and `ALLOW_SYNTHETIC_MATCHES` are explicitly enabled. Shared source files always fail. Invalid reports, empty comparisons, inconsistent statuses and unexpected process failures also fail; synthetic mode never excuses a broken check.

In [ ]:
def validate_leak(proc, synthetic, allow_matches, seq_len, sampled, span):
    try:
        report = json.loads(proc.stdout)
        fields = ("val_sequences_sampled", "val_windows", "train_windows", "matching_windows", "val_sequences_with_match")
        if not isinstance(report, dict) or any(type(report[key]) is not int for key in fields):
            raise ValueError("invalid counters")
        if type(report["leaked"]) is not bool or not isinstance(report["shared_source_files"], list):
            raise ValueError("invalid status")
        if report["seq_len"] != seq_len or report["span"] != span or report["val_sequences_sampled"] != sampled:
            raise ValueError("mismatched sampling configuration")
        if report["val_windows"] != sampled * (seq_len - span + 1) or report["train_windows"] < 1:
            raise ValueError("empty or inconsistent comparison")
        if not 0 <= report["matching_windows"] <= report["val_windows"]:
            raise ValueError("invalid match count")
        if not 0 <= report["val_sequences_with_match"] <= sampled:
            raise ValueError("invalid matched sequence count")
        if bool(report["matching_windows"]) != bool(report["val_sequences_with_match"]):
            raise ValueError("inconsistent matches")
        leaked = bool(report["matching_windows"] or report["shared_source_files"])
        if report["leaked"] != leaked or proc.returncode != int(leaked):
            raise ValueError("exit status disagrees with report")
    except (KeyError, TypeError, ValueError) as error:
        raise RuntimeError(f"Invalid leakage report (exit {proc.returncode}): {proc.stderr}") from error
    if report["shared_source_files"]:
        raise RuntimeError("Train/validation source files overlap")
    if leaked and not (synthetic and allow_matches):
        raise RuntimeError("Validation matches training; refusing training")
    return report


leak_span = min(50, SEQ_LEN)
leak_samples = min(100, prepared["val"])
leak_proc = run([PY, "-m", "brainlab.data.leakage", "--train-dir", DATA / "train", "--val-dir", DATA / "val",
                 "--seq-len", SEQ_LEN, "--sequences", leak_samples, "--span", leak_span, "--seed", SEED, "--json"],
                cwd=REPO, check=False)
leak = validate_leak(leak_proc, DATA_MODE == "synthetic", ALLOW_SYNTHETIC_MATCHES, SEQ_LEN, leak_samples, leak_span)
print(json.dumps(leak, indent=2))
if leak["leaked"]:
    print("Explicit synthetic-only exception: these metrics cannot establish language-model quality")

## Overfit preflight

A separate, fresh temporary run repeats one random sequence (up to 128 tokens by default), and must reduce loss below 1.0. This tests learning, not the memory footprint of the main batch. Packing runs through the selected interpreter. Nonzero training exits always fail even if earlier losses looked good. The next probe uses the actual main batch.

In [ ]:
def train_command(out, train_dir, val_dir, steps, seq_len, micro_batch, accum, eval_every, eval_batches, save_every, log_every):
    command = [PY, "-u", "-m", "brainlab.train.train", "--train-dir", train_dir, "--val-dir", val_dir,
               "--out", out, "--device", DEVICE, "--dtype", DTYPE, "--seq-len", seq_len,
               "--micro-batch", micro_batch, "--accum", accum, "--seed", SEED, "--steps", steps,
               "--eval-every", eval_every, "--eval-batches", eval_batches, "--save-every", save_every,
               "--log-every", log_every, "--keep", KEEP]
    if GRAD_CKPT:
        command.append("--grad-ckpt")
    return command


def pack_overfit(directory, width, seed):
    return py("""
from pathlib import Path
import sys, numpy as np
from brainlab.data.pack import PackConfig, pack_and_write
root, width, seed = Path(sys.argv[1]), int(sys.argv[2]), int(sys.argv[3])
rng = np.random.default_rng(seed)
cfg = PackConfig(seq_len=width, shard_tokens=width * 8, eos_id=257)
pack_and_write([rng.integers(2, 250, size=2 * width - 1).tolist()], root / 'train', 'train', cfg)
pack_and_write([rng.integers(2, 250, size=8 * width - 1).tolist()], root / 'val', 'val', cfg)
""", directory, width, seed)


if RUN_OVERFIT:
    with tempfile.TemporaryDirectory(prefix="overfit-", dir=WORK) as temp:
        overfit_dir = Path(temp)
        width = min(SEQ_LEN, OVERFIT_SEQ_LEN)
        pack_overfit(overfit_dir, width, SEED)
        result = run(train_command(overfit_dir / "run", overfit_dir / "train", overfit_dir / "val",
                                   OVERFIT_STEPS, width, 1, 1, OVERFIT_STEPS, 1, OVERFIT_STEPS, 1), cwd=REPO)
        rows = [row for row in read_rows(overfit_dir / "run") if row.get("event") == "train"]
        if not rows or rows[-1]["step"] != OVERFIT_STEPS or not math.isfinite(rows[-1]["loss"]) or rows[-1]["loss"] >= 1.0:
            raise RuntimeError("Overfit failed to complete and memorise the isolated sequence")
        print("Overfit loss:", rows[0]["loss"], "->", rows[-1]["loss"], "on", DEVICE, hardware["dtype"])
else:
    print("Overfit skipped explicitly; learning preflight not measured")

## Configured-batch probe

The probe uses the configured sequence, micro-batch, accumulation, dtype and checkpointing in a fresh temporary run. A short GPU rate can guide budgeting but is not a validated capacity or end-to-end completion estimate. CPU execution is a functional check only; no throughput conclusion is drawn. Probe and overfit are separate from the main run's budget.

In [ ]:
if RUN_PROBE:
    with tempfile.TemporaryDirectory(prefix="probe-", dir=WORK) as temp:
        probe_dir = Path(temp) / "run"
        result = run(train_command(probe_dir, DATA / "train", DATA / "val", PROBE_STEPS,
                                   SEQ_LEN, MICRO_BATCH, ACCUM, PROBE_STEPS, VAL_BATCHES, PROBE_STEPS, 1), cwd=REPO)
        rows = [row for row in read_rows(probe_dir) if row.get("event") == "train"]
        if not rows or rows[-1]["step"] != PROBE_STEPS or any(not math.isfinite(row["loss"]) for row in rows):
            raise RuntimeError("Configured-batch probe did not complete successfully")
        tokens_per_step = MICRO_BATCH * ACCUM * SEQ_LEN
        if DEVICE == "cpu":
            print("CPU configured-batch functional probe completed; no performance conclusions")
        else:
            rates = [row["tokens_per_sec"] for row in rows if row.get("tokens_per_sec", 0) > 0]
            if not rates:
                raise RuntimeError("Probe has no usable rate measurements")
            rate = statistics.median(rates)
            print("Short GPU probe tokens/s:", rate, "tokens/step:", tokens_per_step)
            print("Training-only hours for configured total:", TOTAL_STEPS * tokens_per_step / rate / 3600)
            print("Excludes setup, evaluation, saving and backups; not a completion guarantee")
else:
    print("Probe skipped explicitly; main-batch fit and throughput unmeasured")

## Budgeted training

Resume may change device/dtype, not data, tokenizer, batching, seed, total schedule or evaluation windows. The notebook contract and trainer checkpoint checks enforce complementary parts of this rule. The child writes locally; stdout and authoritative logs append rather than overwrite.

A budget timeout is intentional; any other nonzero exit fails. There is no assumption that termination creates `last.pt`. Resume uses the latest completed shared cursor and may replay work since its save. If no checkpoint exists, the next attempt starts from zero. Final backup cleanup also runs on errors and KeyboardInterrupt, but a hard runtime kill can prevent it.

In [ ]:
def stop_child(child):
    if child.poll() is not None:
        return child.returncode
    child.terminate()
    try:
        return child.wait(timeout=30)
    except subprocess.TimeoutExpired:
        child.kill()
        return child.wait()


def run_budgeted(command, directory, minutes, backup=None, backup_seconds=300):
    if minutes <= 0 or backup_seconds <= 0:
        raise ValueError("Budget and backup interval must be positive")
    deadline = time.monotonic() + minutes * 60
    next_backup = time.monotonic() + backup_seconds
    child = None
    budget_hit = False
    with (Path(directory) / "train_stdout.log").open("a", encoding="utf-8") as handle:
        try:
            child = subprocess.Popen([str(arg) for arg in command], cwd=str(REPO),
                                     stdout=handle, stderr=subprocess.STDOUT, text=True)
            while True:
                if child.poll() is not None:
                    returncode = child.returncode
                    break
                now = time.monotonic()
                if now >= deadline:
                    budget_hit = True
                    returncode = stop_child(child)
                    break
                if backup is not None and now >= next_backup:
                    handle.flush()
                    backup()
                    next_backup = time.monotonic() + backup_seconds
                    continue
                timeout = deadline - now
                if backup is not None:
                    timeout = min(timeout, next_backup - now)
                try:
                    returncode = child.wait(timeout=timeout)
                    break
                except subprocess.TimeoutExpired:
                    continue
        finally:
            try:
                if child is not None:
                    stop_child(child)
            finally:
                handle.flush()
                if backup is not None:
                    backup()
    if returncode != 0 and not budget_hit:
        raise subprocess.CalledProcessError(returncode, command)
    return {"budget_hit": budget_hit, "returncode": returncode}

In [ ]:
verify_contract(CKPT, RUN_CONTRACT)
prior = RESTORED_CURSOR if RESTORED_CURSOR is not None else latest_cursor(CKPT)
if prior is not None and not prior.is_file():
    raise FileNotFoundError(prior)
command = train_command(CKPT, DATA / "train", DATA / "val", TOTAL_STEPS,
                        SEQ_LEN, MICRO_BATCH, ACCUM, EVAL_EVERY, VAL_BATCHES, SAVE_EVERY, LOG_EVERY)
if prior is not None:
    command += ["--resume", prior]
print("resume:", prior)
backup = (lambda: backup_run(CKPT, BACKUP_DIR)) if BACKUP_DIR is not None else None
try:
    outcome = run_budgeted(command, CKPT, SESSION_MINUTES, backup, BACKUP_SECONDS)
    print(outcome)
    if BACKUP_DIR is not None and not outcome["budget_hit"]:
        backup_run(CKPT, BACKUP_DIR, include_exports=True)
finally:
    RESTORED_CURSOR = None
    checkpoint = latest_cursor(CKPT)
    print("Latest completed cursor:", checkpoint)
    if checkpoint is None:
        print("No checkpoint saved; another attempt would restart at zero, not resume from a presumed last.pt")
    print("Authoritative log:", CKPT / "train_log.jsonl")

In [ ]:
def latest_header(rows):
    headers = [row for row in rows if row.get("event") == "config"]
    if not headers:
        raise RuntimeError("No authoritative config header")
    header = headers[-1]
    track, select = header.get("eval_track_window"), header.get("eval_select_window")
    if not all(isinstance(window, list) and len(window) == 2 and all(type(value) is int for value in window)
               for window in (track, select)):
        raise RuntimeError("Missing or invalid evaluation windows")
    if type(header.get("val_sequences")) is not int or not 0 <= track[0] < track[1] <= select[0] < select[1] <= header["val_sequences"]:
        raise RuntimeError("Evaluation windows must be nonempty, disjoint and within validation data")
    return header


rows = read_rows(CKPT)
header = latest_header(rows)
for key in ("git", "device", "device_name", "dtype", "resumed_from", "start_step", "start_batch",
            "seq_len", "micro_batch", "accum_steps", "total_steps", "seed", "peak_lrs",
            "eval_track_window", "eval_select_window", "report_scope"):
    print(key, header.get(key))
trains = [row for row in rows if row.get("event") == "train"]
if not trains:
    raise RuntimeError("No training records; no learning curve to report")
print("Logged loss:", trains[0]["loss"], "->", trains[-1]["loss"])
print("Logs may include unsaved steps replayed after interruption; checkpoint cursor is authoritative")
for subset in ("track", "select"):
    evaluations = [row for row in rows if row.get("event") == "eval" and row.get("subset") == subset]
    if evaluations:
        print(subset, "latest loss:", evaluations[-1]["val_loss"], "step:", evaluations[-1]["step"])

## Export, evaluate and generate

Export fp32 weights from the selection winner, or the shared latest cursor if there is no winner. Perplexity uses the configured micro-batch and sequence length; `PPL_BATCHES=None` requests all full batches (a final incomplete batch is dropped by the evaluator). Synthetic/charlevel results remain smoke-only.

Optional `COMPARE_FP16` compares **whole-model fp32 versus fp16 autocast**, not a pure output-head ablation. CPU does not measure this comparison. Generation runs one prompt at a time and bounds prompt plus new tokens to the configured sequence length; its CLI has no batch or sequence-length flags.

In [ ]:
chosen = CKPT / "best.pt" if (CKPT / "best.pt").is_file() else latest_cursor(CKPT)
if chosen is None:
    raise RuntimeError("No completed checkpoint to export")
safetensors_out = CKPT / "scored.safetensors"
export_staging = safetensors_out.with_name(safetensors_out.name + ".partial")
try:
    print(run([PY, "-m", "brainlab.export.to_safetensors", "--checkpoint", chosen,
               "--out", export_staging, "--dtype", "f32"], cwd=REPO).stdout)
    export_staging.replace(safetensors_out)
finally:
    export_staging.unlink(missing_ok=True)
    if BACKUP_DIR is not None:
        backup_run(CKPT, BACKUP_DIR, include_exports=True)
print("Exported:", safetensors_out, "from", chosen)

In [ ]:
ppl_command = [PY, "-m", "brainlab.eval.perplexity", "--model", safetensors_out,
               "--data-dir", DATA / "val", "--device", DEVICE, "--batch", MICRO_BATCH, "--seq-len", SEQ_LEN, "--json"]
if PPL_BATCHES is not None:
    if type(PPL_BATCHES) is not int or PPL_BATCHES < 1:
        raise ValueError("PPL_BATCHES must be None or a positive integer")
    ppl_command += ["--batches", PPL_BATCHES]
fp32_row = json.loads(run(ppl_command, cwd=REPO).stdout)
print("fp32:", fp32_row)
print("Evaluation token coverage:", fp32_row["tokens"], "of", prepared["val"] * SEQ_LEN)
if COMPARE_FP16 and DEVICE == "cuda":
    amp_row = json.loads(run([*ppl_command, "--amp"], cwd=REPO).stdout)
    print("fp16 autocast:", amp_row)
    if fp32_row.get("logits_dtype") == amp_row.get("logits_dtype"):
        print("NOT MEASURED: logits dtypes were identical")
    else:
        print("Whole-model autocast loss gap:", abs(fp32_row["loss"] - amp_row["loss"]), "nats; not a head ablation")
else:
    print("Whole-model mixed-precision comparison NOT MEASURED")

In [ ]:
for prompt in PROMPTS:
    prompt_tokens = int(py("""
import sys
from pathlib import Path
from brainlab.data.prepare import byte_tokenizer
from brainlab.tok.reference import Tokenizer
tok = Tokenizer.load(Path(sys.argv[1])) if sys.argv[1] else byte_tokenizer()
print(len(tok.encode(sys.argv[2])))
""", str(Path(TOKENIZER_DIR).expanduser().resolve()) if TOKENIZER_DIR is not None else "", prompt).stdout)
    max_new = min(GENERATE_TOKENS, SEQ_LEN - prompt_tokens)
    if prompt_tokens < 1 or max_new < 1:
        raise ValueError("Prompt must leave room for generation inside SEQ_LEN")
    result = run([PY, "-m", "brainlab.eval.generate", "--model", safetensors_out, *TOKENIZER_ARGS,
                  "--prompt", prompt, "--max-new", max_new, "--temperature", 0.8, "--top-p", 0.9,
                  "--top-k", 40, "--seed", SEED, "--device", DEVICE], cwd=REPO)
    print(result.stdout)

## Keep the evidence

Save the run directory, prepared data and tokenizer as outputs you can explicitly attach for a future session. Keep `train_log.jsonl`, `notebook_run.json`, checkpoints and the scored export together. No automatic Kaggle restore is implied. A short probe, synthetic overfit, or passing subset of tests establishes neither convergence nor hosted GPU performance.

In [ ]:
for item in artifact_paths(CKPT, include_exports=True):
    print(item.name, round(item.stat().st_size / 1e6, 1), "MB")
print("Prepared data to preserve:", DATA)
print("Tokenizer to preserve:", TOKENIZER_DIR or "charlevel (smoke only)")
print("Do not stage or commit generated artifacts")